# Try the trained model

Loads `model/spam_pipeline.joblib` (created by `train-model.ipynb`) and classifies example
emails, the same way `src/app.py` does.

In [ ]:
import sys

import pandas as pd

sys.path.insert(0, "src")
from spam_model import load_model, preprocess_text, spam_probability  # noqa: E402

bundle = load_model("model")
pipeline, threshold = bundle["pipeline"], bundle["threshold"]
print(f"scikit-learn {bundle['sklearn_version']}, threshold {threshold:.3f}")
bundle["metrics"]

## Classify example emails

Edit or add rows to test your own examples. The threshold comes from training; anything at
or above it is flagged as spam.

Watch the *"Your order has shipped"* example: a model trained only on the 2000–2008 public
corpora tends to flag modern transactional email like this, because those corpora contain
almost none of it. Training with your own Gmail export (`src/export_training_data.py`) is the fix.

In [ ]:
examples = [
    ("URGENT: You have won a prize!",
     "Congratulations, you have been selected to win a free vacation. Click here to claim now."),
    ("Your account has been suspended",
     "We detected unusual sign-in activity. Verify your identity within 24 hours at "
     "http://secure-login.example.com/verify or your account will be closed."),
    ("Invoice #48213 overdue",
     "Please see the attached invoice and wire $4,850 to the new bank account below today."),
    ("Meeting Reminder",
     "Hi team, just a reminder that our quarterly review meeting is scheduled for tomorrow at 10 AM. "
     "Please find the agenda attached."),
    ("Your order has shipped",
     "Good news! Your order #1123-554 is on its way and should arrive on Thursday. Track it in your account."),
    ("Lunch on Friday?",
     "Hey, are you free for lunch on Friday? There's a new ramen place near the office."),
]

rows = []
for subject, body in examples:
    probability = spam_probability(pipeline, subject, body)
    rows.append({"subject": subject, "spam probability": probability,
                 "verdict": "SPAM" if probability >= threshold else "ok"})
pd.DataFrame(rows).style.format({"spam probability": "{:.1%}"})

## Why did it decide that?

For linear models, each word's contribution is its TF-IDF value times its weight. Positive
contributions push toward spam.

In [ ]:
def explain(subject, body, top=10):
    vectorizer, clf = pipeline.named_steps["tfidf"], pipeline.named_steps["clf"]
    if not hasattr(clf, "coef_"):
        print("This model has no per-word weights to explain.")
        return None
    vector = vectorizer.transform([f"{subject} {body}"])
    words = vectorizer.get_feature_names_out()[vector.indices]
    contributions = pd.Series(vector.data * clf.coef_[0][vector.indices], index=words)
    return contributions.sort_values(key=abs, ascending=False).head(top).to_frame("contribution")

print(preprocess_text(examples[1][1]))
explain(*examples[1])